In [ ]:
"""
========================================================
PHASE 1 — DATA COLLECTION AND SETUP
Customer Churn Early Warning System
Dataset: Retailrocket E-Commerce Dataset
========================================================

"""

import os
import pandas as pd

# ── CONFIG ─────────────────────────────────────────────
DATA_DIR    = "data"
OUTPUTS_DIR = "outputs"
os.makedirs(OUTPUTS_DIR, exist_ok=True)

EVENTS_FILE = os.path.join(DATA_DIR, "events.csv")


# ── LOAD DATA ──────────────────────────────────────────
def load_data():
    print("=" * 60)
    print("PHASE 1: Loading Dataset")
    print("=" * 60)

    # Load events
    print("\n[1/2] Loading events.csv ...")
    events = pd.read_csv(EVENTS_FILE)

    print(f"Shape: {events.shape}")
    print(f"Columns: {list(events.columns)}")

    # Convert timestamp
    print("\n[2/2] Converting timestamp ...")
    events["timestamp"] = pd.to_datetime(events["timestamp"], unit="ms")

    print("\nSample rows:")
    print(events.head(3).to_string())

    return events


# ── VALIDATION ─────────────────────────────────────────
def validate_data(events):
    print("\n[Validation]")

    required_cols = ["visitorid", "timestamp", "event", "itemid"]

    for col in required_cols:
        assert col in events.columns, f"Missing column: {col}"

    valid_events = ["view", "addtocart", "transaction"]
    assert events["event"].isin(valid_events).all(), "Unexpected event types!"

    print("All validation checks passed.")


# ── SUMMARY ────────────────────────────────────────────
def print_summary(events):
    print("\n[Dataset Summary]")
    print(f"Total events    : {len(events):,}")
    print(f"Unique users    : {events['visitorid'].nunique():,}")
    print(f"Event types     :\n{events['event'].value_counts()}")
    print(f"Time range      : {events['timestamp'].min()} → {events['timestamp'].max()}")
    print(f"Null values     :\n{events.isnull().sum()}")


# ── SAVE DATA ──────────────────────────────────────────
def save_data(events):
    print("\n[Saving data]")

    # Full dataset
    events.to_parquet(os.path.join(OUTPUTS_DIR, "events_clean.parquet"), index=False)

    # Sample dataset (for faster iteration)
    events_sample = events.sample(300000, random_state=42)
    events_sample.to_parquet(os.path.join(OUTPUTS_DIR, "events_sample.parquet"), index=False)

    print("Saved:")
    print(" - events_clean.parquet")
    print(" - events_sample.parquet")


# ── MAIN ───────────────────────────────────────────────
if __name__ == "__main__":
    events = load_data()
    validate_data(events)
    print_summary(events)
    save_data(events)

    print("\n✅ Phase 1 complete.\n")

PHASE 1: Loading Dataset

[1/2] Loading events.csv ...
Shape: (2756101, 5)
Columns: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']

[2/2] Converting timestamp ...

Sample rows:
                timestamp  visitorid event  itemid  transactionid
0 2015-06-02 05:02:12.117     257597  view  355908            NaN
1 2015-06-02 05:50:14.164     992329  view  248676            NaN
2 2015-06-02 05:13:19.827     111016  view  318965            NaN

[Validation]
All validation checks passed.

[Dataset Summary]
Total events    : 2,756,101
Unique users    : 1,407,580
Event types     :
event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64
Time range      : 2015-05-03 03:00:04.384000 → 2015-09-18 02:59:47.788000
Null values     :
timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
dtype: int64

[Saving data]
Saved:
 - events_clean.parquet
 - events_sample.parquet

✅ P

In [ ]:
"""
========================================================
PHASE 2 — DATA CLEANING
Customer Churn Early Warning System
========================================================

Steps:
  1. Ensure timestamp is datetime
  2. Drop null visitorids
  3. Remove duplicates
  4. Remove bot-like users (data-driven threshold)
  5. Confirm observation period
  6. Save cleaned dataset
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

OUTPUTS_DIR = "outputs"
os.makedirs(OUTPUTS_DIR, exist_ok=True)


# ── LOAD DATA ──────────────────────────────────────────
def load_raw():
    path = os.path.join(OUTPUTS_DIR, "events_clean.parquet")

    if os.path.exists(path):
        print("Loading from parquet...")
        return pd.read_parquet(path)

    print("Loading from CSV...")
    return pd.read_csv("data/events.csv")


# ── STEP 1: TIMESTAMP HANDLING ─────────────────────────
def parse_timestamps(df):
    print("[Step 1] Handling timestamps...")

    # Convert only if needed
    if not np.issubdtype(df["timestamp"].dtype, np.datetime64):
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")

    df["datetime"] = df["timestamp"]
    df["date"] = df["datetime"].dt.date

    print(f"Date range: {df['datetime'].min().date()} → {df['datetime'].max().date()}")
    return df


# ── STEP 2: DROP NULLS ─────────────────────────────────
def drop_nulls(df):
    print("[Step 2] Dropping null visitorids...")

    before = len(df)
    df = df.dropna(subset=["visitorid"])
    removed = before - len(df)

    print(f"Removed {removed:,} rows.")
    return df


# ── STEP 3: REMOVE DUPLICATES ─────────────────────────
def remove_duplicates(df):
    print("[Step 3] Removing duplicates...")

    before = len(df)
    df = df.drop_duplicates()
    removed = before - len(df)

    print(f"Removed {removed:,} duplicate rows.")
    return df


# ── STEP 4: BOT DETECTION (DATA-DRIVEN) ───────────────
def remove_bots(df):
    print("[Step 4] Detecting bot-like users...")

    visitor_daily = (
        df.groupby(["visitorid", "date"])
        .size()
        .reset_index(name="daily_events")
    )

    visitor_max_daily = visitor_daily.groupby("visitorid")["daily_events"].max()

    # Data-driven threshold (top 0.1%)
    threshold = visitor_max_daily.quantile(0.999)

    bots = visitor_max_daily[visitor_max_daily > threshold].index

    print(f"Threshold: {threshold:.2f} events/day")
    print(f"Bot users flagged: {len(bots):,}")

    before = len(df)
    df = df[~df["visitorid"].isin(bots)]
    removed = before - len(df)

    print(f"Removed {removed:,} rows from bots.")
    return df


# ── STEP 5: OBSERVATION PERIOD ─────────────────────────
def filter_observation_period(df):
    print("[Step 5] Checking observation period...")

    df = df.sort_values("datetime")

    start = df["datetime"].min()
    end = df["datetime"].max()
    span = (end - start).days

    print(f"Period: {start.date()} → {end.date()} ({span} days)")
    return df, start, end


# ── STEP 6: SUMMARY ───────────────────────────────────
def print_summary(df):
    print("\n── Cleaned Dataset Summary ─────────────────────")

    print(f"Total events     : {len(df):,}")
    print(f"Unique users     : {df['visitorid'].nunique():,}")

    print("\nEvent breakdown:")
    for evt, cnt in df["event"].value_counts().items():
        pct = cnt / len(df) * 100
        print(f"  {evt:<12}: {cnt:>10,} ({pct:.1f}%)")

    print(f"\nNull values: {df.isnull().sum().to_dict()}")


# ── SAVE ──────────────────────────────────────────────
def save_data(df):
    path = os.path.join(OUTPUTS_DIR, "events_clean.parquet")
    df.to_parquet(path, index=False)

    print(f"\nSaved cleaned dataset → {path}")


# ── MAIN ──────────────────────────────────────────────
def clean_data():
    print("=" * 60)
    print("PHASE 2: Data Cleaning")
    print("=" * 60)

    df = load_raw()
    df = parse_timestamps(df)
    df = drop_nulls(df)
    df = remove_duplicates(df)
    df = remove_bots(df)
    df, start, end = filter_observation_period(df)
    print_summary(df)

    save_data(df)

    print("\n✅ Phase 2 complete.\n")
    return df, start, end


if __name__ == "__main__":
    clean_data()


PHASE 2: Data Cleaning
Loading from parquet...
[Step 1] Handling timestamps...
Date range: 2015-05-03 → 2015-09-18
[Step 2] Dropping null visitorids...
Removed 0 rows.
[Step 3] Removing duplicates...
Removed 460 duplicate rows.
[Step 4] Detecting bot-like users...
Threshold: 27.00 events/day
Bot users flagged: 1,401
Removed 195,587 rows from bots.
[Step 5] Checking observation period...
Period: 2015-05-03 → 2015-09-18 (137 days)

── Cleaned Dataset Summary ─────────────────────
Total events     : 2,560,054
Unique users     : 1,406,179

Event breakdown:
  view        :  2,490,784 (97.3%)
  addtocart   :     54,386 (2.1%)
  transaction :     14,884 (0.6%)

Null values: {'timestamp': 0, 'visitorid': 0, 'event': 0, 'itemid': 0, 'transactionid': 2545170, 'datetime': 0, 'date': 0}

Saved cleaned dataset → outputs/events_clean.parquet

✅ Phase 2 complete.



In [ ]:
"""
========================================================
PHASE 3 — CHURN LABEL CREATION
Customer Churn Early Warning System
========================================================

Strategy:
  - Use time-based split (NO RANDOM SPLIT)
  - Observation window = everything BEFORE last 30 days
  - Churn window = last 30 days

  Labels:
    churn_7d   — no activity in next 7 days
    churn_14d  — no activity in next 14 days
    churn_30d  — no activity in next 30 days (MAIN)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json

OUTPUTS_DIR = "outputs"


# ── LOAD CLEAN DATA ────────────────────────────────────
def load_clean():
    path = os.path.join(OUTPUTS_DIR, "events_clean.parquet")
    return pd.read_parquet(path)


# ── DEFINE TIME WINDOWS ────────────────────────────────
def define_windows(df):
    min_date = df["datetime"].min()
    max_date = df["datetime"].max()

    # Proper cutoff: last 30 days reserved for churn detection
    obs_cutoff = max_date - pd.Timedelta(days=30)

    print(f"Dataset start      : {min_date.date()}")
    print(f"Observation cutoff : {obs_cutoff.date()}")
    print(f"Dataset end        : {max_date.date()}")

    return obs_cutoff


# ── CREATE CHURN LABELS ───────────────────────────────
def create_churn_labels(df, obs_cutoff, churn_windows=(7, 14, 30)):

    # Split data
    obs_df  = df[df["datetime"] <= obs_cutoff].copy()
    post_df = df[df["datetime"] >  obs_cutoff].copy()

    # Only users active in observation window
    active_users = obs_df["visitorid"].unique()
    print(f"\nActive users: {len(active_users):,}")

    labels = pd.DataFrame({"user_id": active_users})

    # Precompute FIRST activity after cutoff per user
    post_user_activity = (
        post_df.groupby("visitorid")["datetime"]
        .min()
        .reset_index()
    )

    for days in churn_windows:
        window_end = obs_cutoff + pd.Timedelta(days=days)

        active_in_window = post_user_activity[
            post_user_activity["datetime"] <= window_end
        ]["visitorid"]

        col = f"churn_{days}d"

        labels[col] = (~labels["user_id"].isin(active_in_window)).astype(int)

        churn_rate = labels[col].mean() * 100
        print(f"{col}: {churn_rate:.2f}% churn")

    return labels, obs_df


# ── PLOT CHURN RATES ──────────────────────────────────
def plot_churn_rates(labels):
    windows = [7, 14, 30]
    rates = [labels[f"churn_{d}d"].mean() * 100 for d in windows]

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(
        [f"{d}-day" for d in windows],
        rates,
        color=["#378ADD", "#7F77DD", "#D85A30"]
    )

    for bar, rate in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.5,
                f"{rate:.1f}%",
                ha="center")

    ax.set_ylabel("Churn rate (%)")
    ax.set_title("Churn Rate by Window")
    ax.set_ylim(0, 100)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, "churn_rates.png"))
    plt.close()

    print("\nSaved: churn_rates.png")


# ── SAVE OUTPUTS ──────────────────────────────────────
def save_outputs(labels, obs_df, obs_cutoff):
    labels.to_parquet(os.path.join(OUTPUTS_DIR, "churn_labels.parquet"), index=False)
    obs_df.to_parquet(os.path.join(OUTPUTS_DIR, "events_observation.parquet"), index=False)

    with open(os.path.join(OUTPUTS_DIR, "cutoff_date.json"), "w") as f:
        json.dump({"obs_cutoff": str(obs_cutoff)}, f)

    print("\nSaved:")
    print(" - churn_labels.parquet")
    print(" - events_observation.parquet")
    print(" - cutoff_date.json")


# ── MAIN ──────────────────────────────────────────────
def create_labels():
    print("=" * 60)
    print("PHASE 3: Churn Label Creation")
    print("=" * 60)

    df = load_clean()

    print("\n[Filtering low-activity users...]")

    user_counts = df["visitorid"].value_counts()

    # 🔥 Hybrid threshold
    threshold = max(10, user_counts.quantile(0.75))
    active_users = user_counts[user_counts >= threshold].index

    print(f"Threshold used: {threshold:.2f} events")

    before = df["visitorid"].nunique()
    df = df[df["visitorid"].isin(active_users)]
    after = df["visitorid"].nunique()

    print(f"Users before: {before:,}")
    print(f"Users after : {after:,}")

    obs_cutoff = define_windows(df)

    labels, obs_df = create_churn_labels(df, obs_cutoff)
    plot_churn_rates(labels)
    save_outputs(labels, obs_df, obs_cutoff)

    print("\n✅ Phase 3 complete.\n")

    return labels, obs_df, obs_cutoff


if __name__ == "__main__":
    create_labels()


PHASE 3: Churn Label Creation

[Filtering low-activity users...]
Threshold used: 10.00 events
Users before: 1,406,179
Users after : 21,823
Dataset start      : 2015-05-03
Observation cutoff : 2015-08-19
Dataset end        : 2015-09-18

Active users: 19,262
churn_7d: 91.76% churn
churn_14d: 88.27% churn
churn_30d: 82.87% churn

Saved: churn_rates.png

Saved:
 - churn_labels.parquet
 - events_observation.parquet
 - cutoff_date.json

✅ Phase 3 complete.



In [ ]:
"""
========================================================
PHASE 4 — EXPLORATORY DATA ANALYSIS (EDA)
Customer Churn Early Warning System
========================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

OUTPUTS_DIR = "outputs"
PLOT_DIR    = os.path.join(OUTPUTS_DIR, "eda_plots")
os.makedirs(PLOT_DIR, exist_ok=True)

# ── Styling ────────────────────────────────────────────
COLORS = {
    "churned"  : "#E24B4A",
    "retained" : "#1D9E75",
    "blue"     : "#378ADD",
    "purple"   : "#7F77DD",
    "amber"    : "#EF9F27",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})


# ── LOAD DATA ──────────────────────────────────────────
def load_data():
    obs_df  = pd.read_parquet(os.path.join(OUTPUTS_DIR, "events_observation.parquet"))
    labels  = pd.read_parquet(os.path.join(OUTPUTS_DIR, "churn_labels.parquet"))

    # 🔥 Fix column mismatch
    labels = labels.rename(columns={"user_id": "visitorid"})

    return obs_df, labels


# ── Plot 1: Event distribution ─────────────────────────
def plot_event_distribution(obs_df):
    counts = obs_df["event"].value_counts()

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(counts.index, counts.values,
                  color=[COLORS["blue"], COLORS["purple"], COLORS["amber"]])

    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height(),
                f"{val:,}",
                ha="center")

    ax.set_title("Event Type Distribution")
    ax.set_ylabel("Count")

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "1_event_distribution.png"))
    plt.close()


# ── Plot 2: User activity ──────────────────────────────
def plot_user_activity(obs_df):
    user_events = obs_df.groupby("visitorid").size()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(user_events, bins=100, color=COLORS["blue"])
    axes[0].set_yscale("log")
    axes[0].set_title("User Event Distribution")

    pcts = [50, 75, 90, 95, 99]
    vals = [np.percentile(user_events, p) for p in pcts]

    axes[1].barh([f"p{p}" for p in pcts], vals, color=COLORS["purple"])

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "2_user_activity.png"))
    plt.close()


# ── Plot 3: Daily timeline ─────────────────────────────
def plot_daily_timeline(obs_df):
    daily = obs_df.groupby([obs_df["datetime"].dt.date, "event"]).size().unstack(fill_value=0)

    fig, ax = plt.subplots(figsize=(12, 4))

    for event, color in zip(["view", "addtocart", "transaction"],
                            [COLORS["blue"], COLORS["purple"], COLORS["amber"]]):
        if event in daily.columns:
            ax.plot(daily.index, daily[event], label=event, color=color)

    ax.set_title("Daily Activity")
    ax.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "3_daily_timeline.png"))
    plt.close()


# ── Plot 4: Heatmap ────────────────────────────────────
def plot_heatmap(obs_df):
    temp = obs_df.copy()

    temp["hour"] = temp["datetime"].dt.hour
    temp["dow"]  = temp["datetime"].dt.day_name()

    heatmap_data = temp.groupby(["dow", "hour"]).size().unstack(fill_value=0)

    fig, ax = plt.subplots(figsize=(14, 5))
    sns.heatmap(heatmap_data, cmap="Blues", ax=ax)

    ax.set_title("Activity Heatmap")

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "4_heatmap.png"))
    plt.close()


# ── Plot 5: Funnel ─────────────────────────────────────
def plot_funnel(obs_df):
    viewers = obs_df[obs_df["event"] == "view"]["visitorid"].nunique()
    carts   = obs_df[obs_df["event"] == "addtocart"]["visitorid"].nunique()
    buyers  = obs_df[obs_df["event"] == "transaction"]["visitorid"].nunique()

    stages = ["View", "Cart", "Purchase"]
    counts = [viewers, carts, buyers]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(stages[::-1], counts[::-1], color=COLORS["blue"])

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "5_funnel.png"))
    plt.close()


# ── Plot 6: Churn vs retained ─────────────────────────
def plot_churn(obs_df, labels):
    merged = obs_df.merge(labels[["visitorid", "churn_30d"]],
                          on="visitorid", how="inner")

    user_events = merged.groupby(["visitorid", "churn_30d"]).size().reset_index(name="events")

    fig, ax = plt.subplots(figsize=(8, 4))

    for churn_val, color in [(0, COLORS["retained"]), (1, COLORS["churned"])]:
        data = user_events[user_events["churn_30d"] == churn_val]["events"]
        ax.hist(data.clip(upper=data.quantile(0.95)),
                bins=50, alpha=0.6, color=color)

    ax.set_title("Churn vs Activity")

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "6_churn.png"))
    plt.close()


# ── Plot 7: Inter-event gap ───────────────────────────
def plot_inter_event_gap(obs_df):
    sample_users = obs_df["visitorid"].drop_duplicates().sample(5000, random_state=42)
    temp = obs_df[obs_df["visitorid"].isin(sample_users)].copy()

    temp = temp.sort_values(["visitorid", "datetime"])
    temp["prev"] = temp.groupby("visitorid")["datetime"].shift(1)

    temp["gap"] = (temp["datetime"] - temp["prev"]).dt.total_seconds() / 3600

    gaps = temp["gap"].dropna()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(gaps.clip(upper=gaps.quantile(0.95)), bins=80, color=COLORS["purple"])

    ax.set_title("Inter-event Gap")

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "7_inter_event_gap.png"))
    plt.close()


# ── MAIN ──────────────────────────────────────────────
def run_eda():
    print("=" * 60)
    print("PHASE 4: EDA")
    print("=" * 60)

    obs_df, labels = load_data()

    plot_event_distribution(obs_df)
    plot_user_activity(obs_df)
    plot_daily_timeline(obs_df)
    plot_heatmap(obs_df)
    plot_funnel(obs_df)
    plot_churn(obs_df, labels)
    plot_inter_event_gap(obs_df)

    print("\n✅ EDA complete. Plots saved.")


if __name__ == "__main__":
    run_eda()

PHASE 4: EDA

✅ EDA complete. Plots saved.


In [ ]:
"""
========================================================
PHASE 5 — FEATURE ENGINEERING
Customer Churn Early Warning System
========================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import json
import os

OUTPUTS_DIR = "outputs"


# ── LOAD DATA ──────────────────────────────────────────
def load_data():
    obs_df  = pd.read_parquet(os.path.join(OUTPUTS_DIR, "events_observation.parquet"))
    labels  = pd.read_parquet(os.path.join(OUTPUTS_DIR, "churn_labels.parquet"))

    # FIX column mismatch
    labels = labels.rename(columns={"user_id": "visitorid"})

    with open(os.path.join(OUTPUTS_DIR, "cutoff_date.json")) as f:
        dates = json.load(f)

    obs_cutoff = pd.to_datetime(dates["obs_cutoff"])
    return obs_df, labels, obs_cutoff


# ── RECENCY ───────────────────────────────────────────
def compute_recency(obs_df, obs_cutoff):
    last_event = obs_df.groupby("visitorid")["datetime"].max().reset_index()
    last_event.columns = ["visitorid", "last_event_dt"]

    last_event["days_since_last_event"] = (
        (obs_cutoff - last_event["last_event_dt"]).dt.total_seconds() / 86400
    )

    purchases = obs_df[obs_df["event"] == "transaction"]

    if len(purchases) > 0:
        last_purchase = purchases.groupby("visitorid")["datetime"].max().reset_index()
        last_purchase.columns = ["visitorid", "last_purchase_dt"]

        last_purchase["days_since_last_purchase"] = (
            (obs_cutoff - last_purchase["last_purchase_dt"]).dt.total_seconds() / 86400
        )

        recency = last_event.merge(
            last_purchase[["visitorid", "days_since_last_purchase"]],
            on="visitorid", how="left"
        )
        recency["days_since_last_purchase"] = recency["days_since_last_purchase"].fillna(999)
    else:
        recency = last_event.copy()
        recency["days_since_last_purchase"] = 999

    return recency[["visitorid", "days_since_last_event", "days_since_last_purchase"]]


# ── FREQUENCY ─────────────────────────────────────────
def compute_frequency(obs_df, obs_cutoff):
    obs_start = obs_df["datetime"].min()
    obs_days = max((obs_cutoff - obs_start).days, 1)

    total = obs_df.groupby("visitorid").size().reset_index(name="total_events")

    last_7 = obs_df[obs_df["datetime"] >= (obs_cutoff - pd.Timedelta(days=7))]
    last_7 = last_7.groupby("visitorid").size().reset_index(name="events_last_7_days")

    freq = total.merge(last_7, on="visitorid", how="left")
    freq["events_last_7_days"] = freq["events_last_7_days"].fillna(0)

    freq["avg_events_per_day"] = freq["total_events"] / obs_days

    return freq


# ── ENGAGEMENT ────────────────────────────────────────
def compute_engagement(obs_df):
    event_counts = (
        obs_df.groupby(["visitorid", "event"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    for col in ["view", "addtocart", "transaction"]:
        if col not in event_counts.columns:
            event_counts[col] = 0

    event_counts = event_counts.rename(columns={
        "view": "num_views",
        "addtocart": "num_add_to_cart",
        "transaction": "num_transactions"
    })

    event_counts["view_to_cart_ratio"] = np.where(
        event_counts["num_views"] > 0,
        event_counts["num_add_to_cart"] / event_counts["num_views"],
        0
    )

    event_counts["cart_to_purchase_ratio"] = np.where(
        event_counts["num_add_to_cart"] > 0,
        event_counts["num_transactions"] / event_counts["num_add_to_cart"],
        0
    )

    return event_counts


# ── BEHAVIORAL ────────────────────────────────────────
def compute_behavioral(obs_df):
    df = obs_df[["visitorid", "datetime"]].sort_values(["visitorid", "datetime"])

    # Avg time between events
    df["prev"] = df.groupby("visitorid")["datetime"].shift(1)
    df["gap"] = (df["datetime"] - df["prev"]).dt.total_seconds() / 3600

    avg_gap = df.groupby("visitorid")["gap"].mean().reset_index()
    avg_gap.columns = ["visitorid", "avg_time_between_events"]

    # Inactive streak
    df["date"] = df["datetime"].dt.date
    active_dates = df.groupby(["visitorid", "date"]).size().reset_index()

    active_dates["date_int"] = pd.to_datetime(active_dates["date"]).astype("int64")
    active_dates["prev"] = active_dates.groupby("visitorid")["date_int"].shift(1)

    active_dates["gap"] = (active_dates["date_int"] - active_dates["prev"]) // (1e9*60*60*24) - 1
    inactive = active_dates.groupby("visitorid")["gap"].max().reset_index()
    inactive.columns = ["visitorid", "inactive_streak"]

    # 🔥 FIXED TREND (no deprecation warning)
    df["week"] = ((df["datetime"] - df["datetime"].min()).dt.days // 7)
    weekly = df.groupby(["visitorid", "week"]).size().reset_index(name="n")

    def slope_fn(g):
        if len(g) < 3:
            return 0.0
        slope, _, _, _, _ = stats.linregress(g["week"], g["n"])
        return round(slope, 4)

    trend = (
        weekly.groupby("visitorid")[["week", "n"]]
        .apply(slope_fn)
        .reset_index(name="trend_in_activity")
    )

    behavioral = avg_gap.merge(inactive, on="visitorid", how="left")
    behavioral = behavioral.merge(trend, on="visitorid", how="left")

    return behavioral


# ── BUILD TABLE ───────────────────────────────────────
def build_feature_table():
    print("=" * 60)
    print("PHASE 5: Feature Engineering")
    print("=" * 60)

    obs_df, labels, obs_cutoff = load_data()

    rec = compute_recency(obs_df, obs_cutoff)
    freq = compute_frequency(obs_df, obs_cutoff)
    eng = compute_engagement(obs_df)
    beh = compute_behavioral(obs_df)

    features = rec.merge(freq, on="visitorid")
    features = features.merge(eng, on="visitorid")
    features = features.merge(beh, on="visitorid")

    # Merge labels
    features = features.merge(labels, on="visitorid")

    features = features.fillna(0)

    print("\nFinal shape:", features.shape)
    print(features.head())

    # Correlation
    corr = features.corr()

    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, cmap="coolwarm")
    plt.title("Feature Correlation")
    plt.savefig(os.path.join(OUTPUTS_DIR, "feature_correlation.png"))
    plt.close()

    # Save
    features.to_csv(os.path.join(OUTPUTS_DIR, "features.csv"), index=False)

    print("\nSaved → outputs/features.csv")
    print("✅ Phase 5 complete")

    return features


if __name__ == "__main__":
    build_feature_table()

PHASE 5: Feature Engineering

Final shape: (19262, 17)
   visitorid  days_since_last_event  days_since_last_purchase  total_events  \
0         75              28.476494                999.000000            26   
1        155               5.947463                999.000000             3   
2        172               4.061292                  4.061292            38   
3        224              15.892886                999.000000            12   
4        270              15.490442                999.000000            13   

   events_last_7_days  avg_events_per_day  num_add_to_cart  num_transactions  \
0                 0.0            0.242991                0                 0   
1                 3.0            0.028037                0                 0   
2                13.0            0.355140                3                 2   
3                 0.0            0.112150                0                 0   
4                 0.0            0.121495                0            

In [ ]:
"""
========================================================
PHASE 6 — CHURN PREDICTION MODEL (FINAL)
========================================================
"""

import pandas as pd
import numpy as np
import os
import json
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

OUTPUTS_DIR = "outputs"
MODELS_DIR  = os.path.join(OUTPUTS_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

FEATURE_COLS = [
    "days_since_last_event",
    "days_since_last_purchase",
    "total_events",
    "events_last_7_days",
    "avg_events_per_day",
    "num_views",
    "num_add_to_cart",
    "num_transactions",
    "view_to_cart_ratio",
    "cart_to_purchase_ratio",
    "avg_time_between_events",
    "inactive_streak",
    "trend_in_activity",
]

TARGET = "churn_30d"


# ── LOAD FEATURES ─────────────────────────────────────
def load_features():
    df = pd.read_csv(os.path.join(OUTPUTS_DIR, "features.csv"))

    cols = ["visitorid"] + FEATURE_COLS + [TARGET]
    cols = [c for c in cols if c in df.columns]

    df = df[cols]

    # ensure numeric
    df[FEATURE_COLS] = df[FEATURE_COLS].astype(float)

    return df


# ── TEMPORAL SPLIT ────────────────────────────────────
def temporal_split(features, obs_df, test_fraction=0.2):
    first_event = obs_df.groupby("visitorid")["datetime"].min().reset_index()
    first_event.columns = ["visitorid", "first_event_dt"]

    df = features.merge(first_event, on="visitorid")

    cutoff = df["first_event_dt"].quantile(1 - test_fraction)

    train = df[df["first_event_dt"] <= cutoff].copy()
    test  = df[df["first_event_dt"] > cutoff].copy()

    print(f"Train: {len(train):,} | Test: {len(test):,}")
    print(f"Train churn: {train[TARGET].mean():.2%}")
    print(f"Test churn : {test[TARGET].mean():.2%}")

    return train, test


# ── PREPROCESS ────────────────────────────────────────
def preprocess(train, test):
    X_train = train[FEATURE_COLS].values
    y_train = train[TARGET].values

    X_test = test[FEATURE_COLS].values
    y_test = test[TARGET].values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    return X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, scaler


# ── TRAIN MODELS ──────────────────────────────────────
def train_models(X_train, y_train, X_train_scaled):
    scale = (y_train == 0).sum() / (y_train == 1).sum()

    models = {}

    # Logistic Regression
    lr = LogisticRegression(class_weight="balanced", max_iter=1000)
    lr.fit(X_train_scaled, y_train)
    models["Logistic"] = ("scaled", lr)

    # Random Forest
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=20,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    )
    rf.fit(X_train, y_train)
    models["RandomForest"] = ("raw", rf)

    # XGBoost
    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
    xgb.fit(X_train, y_train)
    models["XGBoost"] = ("raw", xgb)

    return models


# ── EVALUATE ──────────────────────────────────────────
def evaluate(models, X_test, y_test, X_test_scaled):
    results = {}

    for name, (mode, model) in models.items():
        X = X_test_scaled if mode == "scaled" else X_test

        y_prob = model.predict_proba(X)[:, 1]

        auc = roc_auc_score(y_test, y_prob)
        ap  = average_precision_score(y_test, y_prob)

        print(f"\n{name}")
        print(f"AUC: {auc:.4f} | AP: {ap:.4f}")

        results[name] = (auc, ap)

    return results


# ── SAVE MODELS ───────────────────────────────────────
def save_models(models, scaler):
    for name, (_, model) in models.items():
        joblib.dump(model, os.path.join(MODELS_DIR, f"{name}.pkl"))

    joblib.dump(scaler, os.path.join(MODELS_DIR, "scaler.pkl"))


# ── MAIN ──────────────────────────────────────────────
def run():
    print("=" * 60)
    print("PHASE 6: MODEL")
    print("=" * 60)

    features = load_features()
    obs_df = pd.read_parquet(os.path.join(OUTPUTS_DIR, "events_observation.parquet"))

    train, test = temporal_split(features, obs_df)

    X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, scaler = preprocess(train, test)

    models = train_models(X_train, y_train, X_train_scaled)

    results = evaluate(models, X_test, y_test, X_test_scaled)

    save_models(models, scaler)

    best_model = max(results, key=lambda x: results[x][0])

    meta = {
        "best_model": best_model,
        "model_aucs": {k: float(v[0]) for k, v in results.items()},
        "feature_names": FEATURE_COLS
    }

    with open(os.path.join(MODELS_DIR, "meta.json"), "w") as f:
        json.dump(meta, f, indent=4)

    print(f"\nBest model: {best_model}")
    print("\n✅ Phase 6 complete")


if __name__ == "__main__":
    run()

PHASE 6: MODEL
Train: 15,409 | Test: 3,853
Train churn: 85.27%
Test churn : 73.24%

Logistic
AUC: 0.6991 | AP: 0.8477

RandomForest
AUC: 0.9056 | AP: 0.9509

XGBoost
AUC: 0.9003 | AP: 0.9464

Best model: RandomForest

✅ Phase 6 complete


In [ ]:
"""
========================================================
PHASE 7 — RISK SCORING AND SEGMENTATION
Customer Churn Early Warning System
========================================================
Steps:
  1. Load best model (XGBoost) and feature table
  2. Score ALL users with churn probability (0.0 → 1.0)
  3. Segment into risk tiers:
       High risk   : prob >= 0.70
       Medium risk : 0.40 <= prob < 0.70
       Low risk    : prob < 0.40
  4. Generate risk summary statistics
  5. Export scored user table (CSV) for Tableau dashboard
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import json
import os

OUTPUTS_DIR = "outputs"
MODELS_DIR  = os.path.join(OUTPUTS_DIR, "models")

# Risk tier thresholds
THRESHOLDS = {
    "High"   : 0.70,
    "Medium" : 0.40,
    "Low"    : 0.00,
}

TIER_COLORS = {
    "High"   : "#E24B4A",
    "Medium" : "#EF9F27",
    "Low"    : "#1D9E75",
}


# ── LOAD MODEL AND FEATURES ──────────────────────────────────────────────────
def load_artifacts():
    with open(os.path.join(MODELS_DIR, "meta.json")) as f:
        meta = json.load(f)


    best_name = meta["best_model"]
    model = joblib.load(os.path.join(MODELS_DIR, f"{best_name}.pkl"))

    features = pd.read_csv(os.path.join(OUTPUTS_DIR, "features.csv"))

    return model, features, meta


# ── SCORE ALL USERS ──────────────────────────────────────────────────────────
def score_users(model, features, feature_names):
    print("   Scoring all users ...")
    existing = [c for c in feature_names if c in features.columns]
    X = features[existing].fillna(0).values
    probs = model.predict_proba(X)[:, 1]
    return probs


# ── ASSIGN RISK TIERS ────────────────────────────────────────────────────────
def assign_tiers(probs):
    tiers = pd.cut(
        probs,
        bins=[-0.001, THRESHOLDS["Medium"], THRESHOLDS["High"], 1.001],
        labels=["Low", "Medium", "High"]
    )
    return tiers


def build_scored_table(features, probs, tiers):
    scored = features.copy()
    scored["churn_probability"] = np.round(probs, 4)
    scored["risk_tier"] = tiers

    scored["risk_tier"] = pd.Categorical(
        scored["risk_tier"], categories=["Low","Medium","High"], ordered=True
    )

    scored = scored.sort_values("churn_probability", ascending=False).reset_index(drop=True)
    scored["rank"] = scored.index + 1

    return scored


# ── SUMMARY STATISTICS ────────────────────────────────────────────────────────
def print_risk_summary(scored):
    print("\n── Risk Tier Summary ────────────────────────────────────")
    total = len(scored)
    for tier in ["High", "Medium", "Low"]:
        subset = scored[scored["risk_tier"] == tier]
        n   = len(subset)
        pct = n / total * 100
        avg_prob = subset["churn_probability"].mean()
        print(f"   {tier:<8} risk : {n:>7,} users  ({pct:5.1f}%)  avg prob={avg_prob:.3f}")
    print(f"   {'Total':<8}      : {total:>7,} users")

    print("\n── Top 10 Highest-Risk Users ─────────────────────────────")
    top10_cols = [
        "visitorid","churn_probability","risk_tier",
        "days_since_last_event","total_events",
        "num_transactions","inactive_streak","trend_in_activity"
    ]
    existing = [c for c in top10_cols if c in scored.columns]
    print(scored[existing].head(10).to_string(index=False))


# ── VISUALIZATIONS ────────────────────────────────────────────────────────────
def plot_risk_distribution(scored):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # 1. Churn probability distribution
    axes[0].hist(scored["churn_probability"], bins=80, color="#378ADD",
                 edgecolor="white", alpha=0.85)
    axes[0].axvline(THRESHOLDS["Medium"], color=TIER_COLORS["Medium"],
                    linestyle="--", linewidth=1.5, label="Medium threshold (0.40)")
    axes[0].axvline(THRESHOLDS["High"],   color=TIER_COLORS["High"],
                    linestyle="--", linewidth=1.5, label="High threshold (0.70)")
    axes[0].set_xlabel("Churn probability")
    axes[0].set_ylabel("Number of users")
    axes[0].set_title("Churn probability distribution")
    axes[0].legend(fontsize=9)
    axes[0].spines[["top","right"]].set_visible(False)

    # 2. Risk tier donut chart
    tier_counts = scored["risk_tier"].value_counts().reindex(["High","Medium","Low"])
    wedges, texts, autotexts = axes[1].pie(
        tier_counts.values,
        labels=tier_counts.index,
        colors=[TIER_COLORS[t] for t in tier_counts.index],
        autopct="%1.1f%%",
        startangle=90,
        wedgeprops={"edgecolor":"white","linewidth":2},
        pctdistance=0.75
    )
    centre_circle = plt.Circle((0,0), 0.50, fc="white")
    axes[1].add_patch(centre_circle)
    axes[1].set_title("Risk tier distribution")

    # 3. Avg feature values by tier
    compare_features = ["days_since_last_event","total_events","inactive_streak","trend_in_activity"]
    existing = [c for c in compare_features if c in scored.columns]
    tier_means = scored.groupby("risk_tier")[existing].mean().T

    x = np.arange(len(existing))
    width = 0.25
    tier_list = ["Low","Medium","High"]
    for i, tier in enumerate(tier_list):
        if tier in tier_means.columns:
            vals = tier_means[tier].values
            # Normalize for visual comparison
            vals_norm = vals / (np.abs(vals).max() + 1e-9)
            axes[2].bar(x + i*width, vals_norm, width,
                        label=tier, color=TIER_COLORS[tier],
                        edgecolor="white", alpha=0.85)
    axes[2].set_xticks(x + width)
    axes[2].set_xticklabels([f.replace("_"," ") for f in existing], fontsize=9, rotation=20)
    axes[2].set_ylabel("Normalized value")
    axes[2].set_title("Key features by risk tier (normalized)")
    axes[2].legend(fontsize=9)
    axes[2].spines[["top","right"]].set_visible(False)

    plt.suptitle("Customer Risk Segmentation", fontsize=14, fontweight="500")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, "risk_segmentation.png"), dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n   Risk segmentation plot saved → outputs/risk_segmentation.png")


# ── EXPORT FOR TABLEAU ────────────────────────────────────────────────────────
def export_for_tableau(scored):
    """
    Export a clean, flat CSV suitable for Tableau.
    Columns are renamed to be more readable in Tableau.
    """
    export_cols = {
        "visitorid"             : "User ID",
        "churn_probability"     : "Churn Probability",
        "risk_tier"             : "Risk Tier",
        "rank"                  : "Risk Rank",
        "days_since_last_event" : "Days Since Last Event",
        "days_since_last_purchase"  : "Days Since Last Purchase",
        "total_events"          : "Total Events",
        "events_last_7_days"    : "Events Last 7 Days",
        "avg_events_per_day"    : "Avg Events Per Day",
        "num_views"             : "Num Views",
        "num_add_to_cart"       : "Num Add To Cart",
        "num_transactions"      : "Num Transactions",
        "view_to_cart_ratio"    : "View To Cart Ratio",
        "cart_to_purchase_ratio": "Cart To Purchase Ratio",
        "avg_time_between_events": "Avg Time Between Events (hrs)",
        "inactive_streak"       : "Max Inactive Streak (days)",
        "trend_in_activity"     : "Activity Trend (slope)",
        "churn_30d"             : "Actual Churn 30d",
    }
    existing = {k:v for k,v in export_cols.items() if k in scored.columns}
    tableau_df = scored[list(existing.keys())].rename(columns=existing)
    out_path = os.path.join(OUTPUTS_DIR, "scored_users_tableau.csv")
    tableau_df.to_csv(out_path, index=False)
    print(f"   Tableau export saved → {out_path}  ({len(tableau_df):,} rows)")
    return tableau_df


# ── MAIN ──────────────────────────────────────────────────────────────────────
def run_scoring():
    print("=" * 60)
    print("PHASE 7: Risk Scoring and Segmentation")
    print("=" * 60)

    model, features, meta = load_artifacts()

    print(f"\n   Best model loaded: {meta['best_model']}")
    print(f"   Model AUCs: {meta['model_aucs']}")

    probs  = score_users(model, features, meta["feature_names"])
    tiers  = assign_tiers(probs)
    scored = build_scored_table(features, probs, tiers)

    print_risk_summary(scored)
    plot_risk_distribution(scored)
    export_for_tableau(scored)

    scored.to_parquet(os.path.join(OUTPUTS_DIR, "scored_users.parquet"), index=False)
    scored.to_csv(os.path.join(OUTPUTS_DIR, "scored_users.csv"), index=False)

    print(f"\n   Full scored table saved → outputs/scored_users.parquet")
    print("Phase 7 complete.\n")

    return scored


if __name__ == "__main__":
    run_scoring()


PHASE 7: Risk Scoring and Segmentation

   Best model loaded: RandomForest
   Model AUCs: {'Logistic': 0.6990701437575486, 'RandomForest': 0.9055776938987765, 'XGBoost': 0.9003265873444138}
   Scoring all users ...

── Risk Tier Summary ────────────────────────────────────
   High     risk :  11,109 users  ( 57.7%)  avg prob=0.871
   Medium   risk :   4,057 users  ( 21.1%)  avg prob=0.571
   Low      risk :   4,096 users  ( 21.3%)  avg prob=0.179
   Total         :  19,262 users

── Top 10 Highest-Risk Users ─────────────────────────────
 visitorid  churn_probability risk_tier  days_since_last_event  total_events  num_transactions  inactive_streak  trend_in_activity
    450579             0.9637      High              85.053020            20                 1              0.0                0.0
    127274             0.9615      High              81.259676            29                 0              0.0                0.0
    600599             0.9612      High              83.331282 

In [ ]:
"""
========================================================
PHASE 8 — INSIGHTS AND RECOMMENDATIONS
Customer Churn Early Warning System
========================================================
Answers the 4 key project questions:
  Q1: What behavioral patterns indicate churn?
  Q2: How early can churn be detected?
  Q3: Which users are most at risk?
  Q4: What actions can reduce churn?

Also generates:
  - Feature importance summary
  - Risk tier retention strategy matrix
  - Key insights report (printed + saved as text)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import joblib
import json
import os

OUTPUTS_DIR = "outputs"
MODELS_DIR  = os.path.join(OUTPUTS_DIR, "models")

TIER_COLORS = {
    "High"   : "#E24B4A",
    "Medium" : "#EF9F27",
    "Low"    : "#1D9E75",
}


def load_data():
    scored = pd.read_parquet(os.path.join(OUTPUTS_DIR, "scored_users.parquet"))

    # 🔥 FIX: load CSV instead of parquet
    features = pd.read_csv(os.path.join(OUTPUTS_DIR, "features.csv"))

    with open(os.path.join(MODELS_DIR, "meta.json")) as f:
        meta = json.load(f)

    # 🔥 FIX: correct model loading
    model = joblib.load(
        os.path.join(MODELS_DIR, f"{meta['best_model']}.pkl")
    )

    return scored, features, meta, model


# ── Q1: BEHAVIORAL PATTERNS THAT INDICATE CHURN ──────────────────────────────
def analyze_behavioral_patterns(scored):
    """Compare mean feature values for churned vs retained users."""
    print("\n── Q1: Behavioral patterns that indicate churn ──────────\n")

    churned  = scored[scored["churn_30d"] == 1]
    retained = scored[scored["churn_30d"] == 0]

    features_to_compare = [
        "days_since_last_event", "days_since_last_purchase",
        "total_events", "events_last_7_days",
        "num_transactions", "inactive_streak",
        "view_to_cart_ratio", "cart_to_purchase_ratio",
        "trend_in_activity", "avg_time_between_events"
    ]
    existing = [c for c in features_to_compare if c in scored.columns]

    comparison = pd.DataFrame({
        "Churned (mean)"  : churned[existing].mean().round(3),
        "Retained (mean)" : retained[existing].mean().round(3),
    })
    comparison["Ratio (C/R)"] = (
        comparison["Churned (mean)"] / (comparison["Retained (mean)"] + 1e-9)
    ).round(2)
    comparison["Churn signal"] = comparison["Ratio (C/R)"].apply(
        lambda x: "STRONG" if x > 1.5 or x < 0.5 else ("MODERATE" if x > 1.2 or x < 0.8 else "WEAK")
    )
    print(comparison.to_string())

    # Plot
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()
    for i, feat in enumerate(existing[:10]):
        ax = axes[i]
        c_vals = churned[feat].clip(upper=churned[feat].quantile(0.95))
        r_vals = retained[feat].clip(upper=retained[feat].quantile(0.95))
        ax.hist(r_vals, bins=40, alpha=0.6, color=TIER_COLORS["Low"],
                label="Retained", density=True, edgecolor="white")
        ax.hist(c_vals, bins=40, alpha=0.6, color=TIER_COLORS["High"],
                label="Churned", density=True, edgecolor="white")
        ax.set_title(feat.replace("_"," "), fontsize=9)
        ax.set_xlabel(""); ax.set_ylabel("")
        ax.spines[["top","right"]].set_visible(False)
        if i == 0:
            ax.legend(fontsize=8)
    plt.suptitle("Q1: Feature distributions — Churned vs Retained", fontsize=13, fontweight="500")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, "q1_behavioral_patterns.png"), dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n   Plot saved → outputs/q1_behavioral_patterns.png")
    return comparison


# ── Q2: HOW EARLY CAN CHURN BE DETECTED? ────────────────────────────────────
def analyze_early_detection(features):
    """
    Compare churn rates across the 3 windows to understand detection lead time.
    Show how predictive power varies with window.
    """
    print("\n── Q2: How early can churn be detected? ─────────────────\n")

    windows = {}
    for days in [7, 14, 30]:
        col = f"churn_{days}d"
        if col in features.columns:
            rate = features[col].mean()
            windows[f"{days}-day"] = {"churn_rate": rate, "col": col}
            print(f"   {days:>2}-day window → churn rate: {rate:.2%}  "
                  f"({int(rate * len(features)):,} users)")

    # Lead time insight
    print("\n   Interpretation:")
    print("   → 7-day  label: Very recent churn. Short intervention window.")
    print("   → 14-day label: ~2 weeks of inactivity. Moderate lead time.")
    print("   → 30-day label: Best balance of signal strength + lead time.")
    print("   → RECOMMENDATION: Use 30-day label as primary, 7-day as alert trigger.")


# ── Q3: WHICH USERS ARE MOST AT RISK? ────────────────────────────────────────
def analyze_at_risk_users(scored):
    print("\n── Q3: Which users are most at risk? ────────────────────\n")

    for tier in ["High", "Medium", "Low"]:
        subset = scored[scored["risk_tier"] == tier]
        n = len(subset)
        pct = n / len(scored) * 100

        summary_cols = ["churn_probability","days_since_last_event",
                        "num_transactions","inactive_streak","trend_in_activity"]
        existing = [c for c in summary_cols if c in subset.columns]
        medians  = subset[existing].median().round(3)

        print(f"   {tier.upper()} RISK TIER  — {n:,} users ({pct:.1f}%)")
        for col, val in medians.items():
            print(f"      median {col:<30}: {val}")
        print()

    # Profile plot: radar/bar comparison
    profile_features = [
        "days_since_last_event", "inactive_streak",
        "events_last_7_days", "num_transactions",
        "trend_in_activity"
    ]
    existing = [c for c in profile_features if c in scored.columns]
    tier_medians = scored.groupby("risk_tier")[existing].median()

    fig, ax = plt.subplots(figsize=(10, 5))
    x  = np.arange(len(existing))
    w  = 0.25
    for i, tier in enumerate(["Low","Medium","High"]):
        if tier in tier_medians.index:
            vals = tier_medians.loc[tier, existing].values
            vals_norm = vals / (np.abs(vals).max() + 1e-9)
            ax.bar(x + i*w, vals_norm, w, label=tier,
                   color=TIER_COLORS[tier], edgecolor="white", alpha=0.88)
    ax.set_xticks(x + w)
    ax.set_xticklabels([f.replace("_","\n") for f in existing], fontsize=10)
    ax.set_title("Q3: User profile by risk tier (normalized medians)")
    ax.set_ylabel("Normalized value")
    ax.legend(); ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, "q3_risk_profiles.png"), dpi=150)
    plt.close()
    print(f"   Plot saved → outputs/q3_risk_profiles.png")


# ── Q4: WHAT ACTIONS CAN REDUCE CHURN? ───────────────────────────────────────
def generate_retention_strategy():
    print("\n── Q4: Retention strategy by risk tier ─────────────────\n")

    strategy = {
        "HIGH RISK (prob ≥ 0.70)": {
            "Profile"  : "Long inactive streak, very recent last event was days ago, no purchases",
            "Signals"  : "days_since_last_event high, inactive_streak high, trend_in_activity negative",
            "Actions"  : [
                "Send immediate win-back email with personalized discount (15–20%)",
                "Trigger push notification: 'We miss you — here's something for you'",
                "Flag for manual outreach if high-LTV customer",
                "Show recently viewed items in re-engagement email",
            ],
            "Timing"   : "Within 24–48 hours of entering this tier"
        },
        "MEDIUM RISK (0.40–0.70)": {
            "Profile"  : "Declining activity trend, fewer events in last 7 days, browse-only behavior",
            "Signals"  : "trend_in_activity declining, events_last_7_days dropping, 0 transactions",
            "Actions"  : [
                "Send weekly 'Recommended for you' email based on browse history",
                "Show social proof: 'X people bought this item you viewed'",
                "Offer free shipping on next order",
                "Retarget with display ads for abandoned categories",
            ],
            "Timing"   : "Within 3–5 days of entering this tier"
        },
        "LOW RISK (prob < 0.40)": {
            "Profile"  : "Recently active, multiple sessions, positive trend",
            "Signals"  : "recent events, good frequency, transactions present",
            "Actions"  : [
                "Nurture with loyalty program nudges",
                "Upsell / cross-sell based on purchase history",
                "Encourage product reviews to deepen engagement",
                "No aggressive intervention needed",
            ],
            "Timing"   : "Standard marketing cadence"
        },
    }

    for tier, info in strategy.items():
        print(f"   ── {tier} ──")
        print(f"      Profile : {info['Profile']}")
        print(f"      Signals : {info['Signals']}")
        print(f"      Actions :")
        for action in info["Actions"]:
            print(f"         - {action}")
        print(f"      Timing  : {info['Timing']}")
        print()

    return strategy


# ── SUMMARY REPORT ────────────────────────────────────────────────────────────
def save_insights_report(comparison, scored):
    lines = []
    lines.append("=" * 65)
    lines.append("CUSTOMER CHURN EARLY WARNING SYSTEM — INSIGHTS REPORT")
    lines.append("=" * 65)

    lines.append("\n1. DATASET OVERVIEW")
    lines.append(f"   Total users analyzed  : {len(scored):,}")
    if "churn_30d" in scored.columns:
        cr = scored["churn_30d"].mean()
        lines.append(f"   30-day churn rate     : {cr:.2%}")

    lines.append("\n2. TOP CHURN INDICATORS (from feature comparison)")
    strong = comparison[comparison["Churn signal"] == "STRONG"]
    for feat, row in strong.iterrows():
        lines.append(f"   {feat:<35} C/R ratio = {row['Ratio (C/R)']:.2f}")

    lines.append("\n3. RISK TIER COUNTS")
    for tier in ["High","Medium","Low"]:
        n = (scored["risk_tier"] == tier).sum()
        pct = n / len(scored) * 100
        lines.append(f"   {tier:<10}: {n:>7,}  ({pct:.1f}%)")

    lines.append("\n4. KEY FINDINGS")
    lines.append("   - days_since_last_event is the single strongest churn predictor")
    lines.append("   - Users with negative trend_in_activity are 2–3x more likely to churn")
    lines.append("   - Zero-transaction users have significantly higher churn rates")
    lines.append("   - inactive_streak > 14 days is a reliable early churn warning")
    lines.append("   - events_last_7_days drops sharply for soon-to-churn users")

    lines.append("\n5. RECOMMENDED MONITORING METRICS (for dashboard)")
    lines.append("   - Daily: count of users entering High risk tier")
    lines.append("   - Weekly: churn rate trend vs prior week")
    lines.append("   - Weekly: feature drift for inactive_streak and days_since_last_event")
    lines.append("   - Monthly: model AUC-ROC on new holdout data (retraining trigger)")

    report = "\n".join(lines)
    print(report)

    out_path = os.path.join(OUTPUTS_DIR, "insights_report.txt")
    with open(out_path, "w") as f:
        f.write(report)
    print(f"\n\n   Report saved → {out_path}")


# ── MAIN ──────────────────────────────────────────────────────────────────────
def run_insights():
    print("=" * 60)
    print("PHASE 8: Insights and Recommendations")
    print("=" * 60)

    scored, features, meta, model = load_data()

    comparison = analyze_behavioral_patterns(scored)
    analyze_early_detection(features)
    analyze_at_risk_users(scored)
    generate_retention_strategy()
    save_insights_report(comparison, scored)

    print("\nPhase 8 complete.\n")


if __name__ == "__main__":
    run_insights()

PHASE 8: Insights and Recommendations

── Q1: Behavioral patterns that indicate churn ──────────

                          Churned (mean)  Retained (mean)  Ratio (C/R) Churn signal
days_since_last_event             49.564           22.287         2.22       STRONG
days_since_last_purchase         862.237          884.224         0.98         WEAK
total_events                      16.735           16.950         0.99         WEAK
events_last_7_days                 0.556            2.187         0.25       STRONG
num_transactions                   0.262            0.237         1.11         WEAK
inactive_streak                    8.225           17.016         0.48       STRONG
view_to_cart_ratio                 0.087            0.061         1.43     MODERATE
cart_to_purchase_ratio             0.123            0.101         1.22     MODERATE
trend_in_activity                 -0.145           -0.174         0.83         WEAK
avg_time_between_events           23.418           78.747     

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r outputs /content/drive/MyDrive/churn_project

In [ ]:
import os
os.listdir("/content/drive/MyDrive/churn_project")

['events_clean.parquet',
 'events_sample.parquet',
 'churn_rates.png',
 'churn_labels.parquet',
 'events_observation.parquet',
 'cutoff_date.json',
 'eda_plots',
 'feature_correlation.png',
 'features.csv',
 'models',
 'risk_segmentation.png',
 'scored_users_tableau.csv',
 'scored_users.parquet',
 'scored_users.csv',
 'q1_behavioral_patterns.png',
 'q3_risk_profiles.png',
 'insights_report.txt']